#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col, regexp_replace
from pyspark.sql.window import Window
import uuid
from datetime import datetime

In [0]:
# erp_px_cat_g1v2_raw time (starts here...)

start_time = datetime.now()

# Reading From Bronze

In [0]:
df = spark.table("salesdatalakehouse.bronze.erp_px_cat_g1v2_raw")

###No Data Transformations, this table has a very good data quality.

## Renaming Colmns

In [0]:
df = df.toDF(
    "category_id",
    "category",
    "subcategory",
    "maintenance"
)

# Load Into Silver

In [0]:
df.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable('salesdatalakehouse.silver.erp_px_cat_g1v2')

#erp_px_cat_g1v2 Logging

In [0]:

# erp_loc_a101_raw time (ends here...)

run_id = str(uuid.uuid4())
job_name = "sales_etl"
task_name = 'load_Silver'
schema_name = 'silver'
table_name = 'erp_px_cat_g1v2'
row_inserted = spark.sql(''' SELECT COUNT(*) FROM salesdatalakehouse.silver.erp_px_cat_g1v2''').collect()[0][0]
end_time = datetime.now()

# Logging into pipeline_runs table
spark.sql(f"""
  INSERT INTO salesdatalakehouse.audit.pipeline_runs
  VALUES (
    '{run_id}',
    '{job_name}',
    '{task_name}',
    '{schema_name}',
    '{table_name}',
    '{start_time}',
    '{end_time}',
    DATEDIFF(SECOND, '{start_time}', '{end_time}'),
    'SUCCESS',
    '{row_inserted}',
    0,
    0,
    NULL,
    CURRENT_TIMESTAMP()
  )
""")

print(f"✓ erp_px_cat_g1v2 info Logged with run {run_id}")